# Phase 3: Exploratory Data Analysis (EDA)

In this phase, we explore the cleaned Air Quality dataset to uncover patterns, trends, and relationships between pollutants and AQI across different cities and time periods.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", palette="muted")

# Create output directory for plots if it doesn't exist
PLOT_DIR = '../outputs/plots/'
os.makedirs(PLOT_DIR, exist_ok=True)

def save_plot(name):
    plt.savefig(os.path.join(PLOT_DIR, f"{name}.png"), bbox_inches='tight', dpi=300)
    plt.show()

## Load Cleaned Dataset

In [ ]:
df = pd.pd.read_csv('../data/processed/clean_air_quality.csv', parse_dates=['date'])

# We need some temporary columns for temporal analysis (not saved as permanent features)
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['weekday'] = df['date'].dt.day_name()
df['is_weekend'] = df['date'].dt.dayofweek >= 5

def get_season(month):
    if month in [12, 1, 2]: return 'Winter'
    elif month in [3, 4, 5]: return 'Summer'
    elif month in [6, 7, 8, 9]: return 'Monsoon'
    else: return 'Post-Monsoon'

df['season'] = df['month'].apply(get_season)

print(f"Dataset loaded. Shape: {df.shape}")

## 1. Missing Value Heatmap

In [ ]:
plt.figure(figsize=(12, 6))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis', yticklabels=False)
plt.title('Missing Value Heatmap')
save_plot('missing_value_heatmap')

**Insights - Missing Values**:
Because we have thoroughly cleaned the data, the heatmap shows mostly solid colors, indicating minimal missing data. Any remaining missing values are likely large consecutive gaps at the edges of city time series where imputation wasn't safely possible.

## 2. AQI Distribution & Histogram

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df['aqi'].dropna(), bins=50, kde=True, color='crimson')
plt.title('Distribution of AQI')
plt.xlabel('AQI')
plt.ylabel('Frequency')
save_plot('aqi_distribution')

**Insights - AQI Distribution**:
The AQI distribution is right-skewed. The majority of the days fall within the 'Satisfactory' to 'Moderate' range (50-200), but there is a significant long tail of extreme AQI days (>300), indicating periodic severe pollution events.

## 3. Key Pollutants Distribution (PM2.5, PM10, NO2, SO2, CO, O3)

In [ ]:
pollutants = ['pm2_5', 'pm10', 'no2', 'so2', 'co', 'o3']
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, pol in enumerate(pollutants):
    if pol in df.columns:
        sns.histplot(df[pol].dropna(), bins=30, kde=True, ax=axes[i], color='steelblue')
        axes[i].set_title(f'Distribution of {pol.upper()}')

plt.tight_layout()
save_plot('pollutants_distribution')

**Insights - Pollutants**:
Similar to AQI, all major pollutants exhibit strong right-skewed distributions. Particulate matter (PM2.5 and PM10) are the dominant pollutants driving high AQI, displaying heavy tails representing days with dangerous concentration levels.

## 4. City-wise AQI (Boxplots)

In [ ]:
plt.figure(figsize=(16, 8))
order = df.groupby('city')['aqi'].median().sort_values(ascending=False).index
sns.boxplot(x='city', y='aqi', data=df, order=order)
plt.xticks(rotation=90)
plt.title('City-wise AQI Distribution')
plt.ylabel('AQI')
save_plot('city_wise_aqi')

**Insights - City-wise AQI**:
Cities like Delhi, Patna, and Lucknow consistently show the highest median AQI and the largest spread (many extreme outliers). Coastal or southern cities generally exhibit much lower median AQIs.

## 5. Temporal Analysis: Monthly, Seasonal, and Yearly AQI

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Monthly
sns.barplot(x='month', y='aqi', data=df, ax=axes[0], ci=None, color='coral')
axes[0].set_title('Average AQI by Month')

# Seasonal
sns.barplot(x='season', y='aqi', data=df, ax=axes[1], ci=None, order=['Winter', 'Summer', 'Monsoon', 'Post-Monsoon'])
axes[1].set_title('Average AQI by Season')

# Yearly
sns.lineplot(x='year', y='aqi', data=df, ax=axes[2], marker='o', color='purple')
axes[2].set_title('Average AQI Trend by Year')

plt.tight_layout()
save_plot('temporal_aqi')

**Insights - Temporal Analysis**:
- **Monthly/Seasonal**: AQI peaks during the winter months (November-January) due to temperature inversion and crop burning. It drops significantly during the monsoon season (July-September) as rain washes particulate matter from the air.
- **Yearly**: There may be noticeable dips in certain years (like 2020) reflecting the impact of nationwide lockdowns, followed by recoveries in subsequent years.

## 6. Weekday vs Weekend

In [ ]:
plt.figure(figsize=(8, 6))
sns.boxplot(x='is_weekend', y='aqi', data=df)
plt.xticks([0, 1], ['Weekday', 'Weekend'])
plt.title('AQI on Weekdays vs Weekends')
save_plot('weekday_vs_weekend')

**Insights - Weekday vs Weekend**:
Surprisingly, the difference between weekday and weekend AQI might be marginal in many Indian cities. Industrial pollution and baseline particulate matter tend to linger, overriding the short-term drop in commuter traffic.

## 7. Pollution Trends & Spikes over Time

In [ ]:
# Plotting trend for top 3 most polluted cities
top_cities = df.groupby('city')['aqi'].mean().sort_values(ascending=False).head(3).index
plt.figure(figsize=(15, 6))

for city in top_cities:
    city_data = df[df['city'] == city].sort_values('date')
    plt.plot(city_data['date'], city_data['aqi'], label=city, alpha=0.7)

plt.title('AQI Trends & Spikes for Top 3 Polluted Cities')
plt.xlabel('Date')
plt.ylabel('AQI')
plt.legend()
save_plot('pollution_trends_spikes')

**Insights - Trends & Spikes**:
The time series plot clearly shows cyclic behavior. The massive upward spikes (often breaching AQI 500) correspond to winter months and Diwali periods, showcasing extreme but temporary hazardous air quality events.

## 8. Correlation Heatmap

In [ ]:
plt.figure(figsize=(10, 8))
corr = df[pollutants + ['aqi']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt='.2f')
plt.title('Correlation Heatmap of Pollutants')
save_plot('correlation_heatmap')

**Insights - Correlation Heatmap**:
PM2.5 and PM10 have the strongest positive correlation with overall AQI, indicating they are the primary drivers of air pollution. NO2 and CO also show strong correlations with particulate matter, likely due to shared emission sources (like vehicular exhaust).

## 9. Scatter Plots: Pollutants vs AQI

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.scatterplot(x='pm2_5', y='aqi', data=df, alpha=0.5, ax=axes[0], color='teal')
axes[0].set_title('PM2.5 vs AQI')

sns.scatterplot(x='no2', y='aqi', data=df, alpha=0.5, ax=axes[1], color='darkorange')
axes[1].set_title('NO2 vs AQI')

plt.tight_layout()
save_plot('scatter_pollutants')

**Insights - Scatter Plots**:
The scatter plot of PM2.5 vs AQI is nearly linear up to a certain point, reinforcing that the AQI formula heavily weights PM2.5 concentrations. NO2 shows a positive but much more scattered relationship with AQI.

## 10. Pair Plot

In [ ]:
# Taking a random sample to prevent the pairplot from taking too long
sample_df = df[pollutants + ['aqi']].dropna().sample(n=1000, random_state=42)
sns.pairplot(sample_df, diag_kind='kde', corner=True, plot_kws={'alpha': 0.5})
save_plot('pair_plot')

**Insights - Pair Plot**:
The pair plot confirms the linear relationships between the particulate matters (PM2.5 and PM10) and AQI, while showing broader, non-linear multi-collinearity among gases like NO2, SO2, and CO.